# GökTürk-Ultra — Kaggle eğitim denemesi

Bu notebook **altyapıyı** çalıştırır ve doğrular. Şunları yapar:

1. Ortamı ve donanımı kontrol eder (GPU, CUDA, VRAM, paket sürümleri).
2. Gerekli paketleri kurar (sürümler açıkça sabitlenmiştir).
3. Veri dosyasını şemaya ve kalite kurallarına göre doğrular.
4. **Duman testi** yapar: rastgele başlatılmış küçük bir modelle 2 adım eğitim.
   Bu, boru hattının çalıştığını kanıtlar; **model kalitesi hakkında hiçbir şey kanıtlamaz**.
5. Gerçek eğitimi dener; donanım/mimari uygun değilse **dürüstçe durur** ve tanılama yazdırır.
6. Çıktıları `/kaggle/working` altına kaydeder.

## Kaggle'da yapılacaklar (notebook'u çalıştırmadan önce)

* Sağ üstte **Accelerator → GPU T4 x2** seçin ve **Internet → On** yapın.
* **Add-ons → Secrets** bölümüne şunları ekleyin (değerleri buraya yazmayın):
  `EVREN_API_KEY`, `EVREN_BASE_URL`, `EVREN_MODEL_MIMO`, `EVREN_MODEL_KIMI`.
* Eğitim verisini (JSONL) bir **Kaggle Dataset** olarak ekleyin; notebook
  `/kaggle/input` altında arar.

## Dürüstlük notu

`openai/gpt-oss-20b` ağırlıkları MXFP4 olarak yayınlanmıştır ve 2×T4 (2×16 GB,
bf16 yok) üzerinde bu repodaki transformers+peft yolu ile eğitilemez.
Notebook bunu gizlemez; engelleyici bulguları yazdırır. Ayrıntı: `docs/architecture.md`.

## 1) Ortam bilgisi

In [ ]:
import os, platform, subprocess, sys, textwrap

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Çalışma dizini:", os.getcwd())
print("CPU çekirdeği:", os.cpu_count())
try:
    with open("/proc/meminfo") as fh:
        for line in fh:
            if line.startswith("MemTotal"):
                print("RAM:", round(int(line.split()[1]) / 1024**2, 1), "GB")
                break
except OSError:
    pass
print("\n--- nvidia-smi ---")

# Metin tabanlı GPU özeti (nvidia-smi yoksa sessizce geç).
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(r.stdout.strip() or r.stderr.strip() or "nvidia-smi bulunamadı")

## 2) Paketler

Sürümler `pyproject.toml` ile uyumludur. `INSTALL=False` yaparak Kaggle
imajındaki mevcut sürümleri kullanabilirsiniz; ancak `transformers>=4.55`
gpt-oss mimarisi (`GptOssForCausalLM`) için gereklidir.

In [ ]:
INSTALL = True  # kurulumu atlamak için False yapın
PINNED = [
    "transformers>=4.55,<6",   # gpt-oss mimarisi (GptOssForCausalLM) için
    "peft>=0.12",
    "accelerate>=0.33",
    "bitsandbytes>=0.43",      # QLoRA (yalnızca CUDA)
    "pyyaml>=6.0",
]

if INSTALL:
    import subprocess, sys
    # %pip yerine subprocess: hücre gerçek Python olarak da çalıştırılabilir.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=False)

import importlib
for name in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "yaml"]:
    try:
        mod = importlib.import_module(name)
        print(f"{name:<14}", getattr(mod, "__version__", "?"))
    except Exception as exc:
        print(f"{name:<14} KURULU DEĞİL ({type(exc).__name__})")


## 3) Depoyu hazırla ve Kaggle Secrets'ı ortama aktar

Secrets yalnızca ortam değişkenine aktarılır; hiçbir hücre anahtarı yazdırmaz
veya dosyaya yazmaz.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/a23521745-hub/GokTurk-Ultra.git"  # gerekirse değiştirin
REPO_DIR = pathlib.Path("/kaggle/working/GokTurk-Ultra")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Depo zaten var:", REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))
print("src yolu eklendi:", REPO_DIR / "src")

# --- Kaggle Secrets -> ortam değişkenleri (değerler yazdırılmaz) ----------
SECRET_KEYS = ["EVREN_API_KEY", "EVREN_BASE_URL", "EVREN_MODEL_MIMO", "EVREN_MODEL_KIMI"]
try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for key in SECRET_KEYS:
        try:
            os.environ[key] = client.get_secret(key)
        except Exception:
            pass  # secret tanımlı değil: eksik olduğunu aşağıda raporluyoruz
except Exception:
    print("kaggle_secrets yok (Kaggle dışında çalışıyor olabilirsiniz)")

for key in SECRET_KEYS:
    print(f"{key:<18}", "TANIMLI" if os.environ.get(key) else "yok")

## 4) Donanım / mimari ön denetimi

`gokturk.env_check` gerçek donanımı okur ve istenen yöntemin desteklenip
desteklenmediğini raporlar. Çıktı **yorumlanmış** bir tanılamadır, tahmin
tablosu değildir.

In [ ]:
from gokturk.env_check import check_environment

report = check_environment(
    model_name="openai/gpt-oss-20b",
    method="qlora",
    dtype="auto",
    load_in_4bit=True,
    seq_len=1024,
)
print(report.render())

print("\n>>> ÖN DENETİM SONUCU:", "GEÇTİ" if report.ok else "ENGELLEYİCİ SORUN VAR")
if not report.ok:
    print("Aşağıdaki adımlar yine de çalıştırılabilir; gerçek eğitim başlatılmayacaktır.")

## 5) Eğitim verisini doğrula

Veri dosyasını `/kaggle/input` altında arar. Şema: `id`, `prompt`, `response`,
`source` (bkz. `src/gokturk/schema.py`). Bozuk satırlar, tekrarlar ve aşırı
uzun kayıtlar raporlanır.

In [ ]:
import json, pathlib

from gokturk.data_quality import build_datasets, read_jsonl

candidates = sorted(pathlib.Path("/kaggle/input").rglob("*.jsonl")) if pathlib.Path("/kaggle/input").exists() else []
print("Bulunan JSONL dosyaları:")
for path in candidates[:20]:
    print("  -", path)

# Elle de belirtebilirsiniz:
DATA_FILE = None  # örn. pathlib.Path("/kaggle/input/gtu-data/train.jsonl")

if DATA_FILE is None:
    preferred = [p for p in candidates if "train" in p.name.lower()] or candidates
    DATA_FILE = preferred[0] if preferred else REPO_DIR / "data" / "samples" / "sample_train.jsonl"

print("\nKullanılacak veri:", DATA_FILE)

records, issues = read_jsonl(DATA_FILE)
print(f"Okunan kayıt: {len(records)} | Ayrıştırma hatası: {len(issues)}")

PROCESSED = pathlib.Path("/kaggle/working/data/processed")
if records:
    try:
        data_report = build_datasets(DATA_FILE, PROCESSED)
        print("\nVeri raporu:", data_report.summary())
    except ValueError as exc:
        print("Veri kalitesi hatası:", exc)
else:
    print("UYARI: veri dosyası boş; duman testi örnek veriyle yapılacak.")

## 6) Duman testi (hızlı, indirme gerektirmez)

Rastgele başlatılmış küçük bir model + dosyadan üretilen kelime düzeyi
tokenizer ile 2 optimizasyon adımı. Amaç: veri yükleme, maskeleme, LoRA,
Trainer ve kaydetme yolunun uçtan uca çalıştığını göstermek.

**Bu bir eğitim değildir ve hiçbir kalite sonucu üretmez.**

In [ ]:
SMOKE_OUT = "/kaggle/working/smoke"
args = ["--smoke", "--max-steps", "2", "--output-dir", SMOKE_OUT]
if DATA_FILE.exists():
    args += ["--train-file", str(DATA_FILE), "--eval-file", str(DATA_FILE)]

# Alt süreç yerine doğrudan fonksiyon çağrısı: hata çıktısı hücrede görünür.
import gokturk.train as gtu_train
from gokturk.config import load_training_config

cfg = load_training_config(REPO_DIR / "configs" / "training.yaml")
if DATA_FILE.exists():
    cfg.data.train_file = str(DATA_FILE)
    cfg.data.eval_file = str(DATA_FILE)

smoke_code = gtu_train.run_training(cfg, smoke=True, max_steps=2, output_dir=SMOKE_OUT)
print("\nDuman testi çıkış kodu:", smoke_code, "(0 = başarılı)")

manifest_path = pathlib.Path(SMOKE_OUT) / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("kind:", manifest["kind"], "| örnek:", manifest["data"]["train_examples"])
    print("not:", manifest["note"])
else:
    print("MANİFEST YOK: duman testi tamamlanmadı, aşağıdaki hata mesajına bakın.")

## 7) Gerçek eğitim denemesi

Bu hücre gerçek modeli indirmeye ve eğitmeye çalışır. 2×T4 + gpt-oss-20b
birleşiminde ön denetim engelleyeceği için **eğitim başlamaz**; nedenini ve
alternatifleri yazar. Bu davranış bilinçlidir: desteklenmeyen bir yöntem
çalışıyormuş gibi gösterilmez.

In [ ]:
TRAIN_OUT = "/kaggle/working/gokturk-qlora-v1"

# Ön denetim geçmezse hiçbir şey indirilmez.
if not report.ok:
    print("Eğitim atlandı: ön denetim engelleyici sorun bildirdi.")
    for finding in report.blocking:
        print(f"  [ENGELLEYİCİ] {finding.code}: {finding.message}")
        if finding.detail:
            print("      ", finding.detail)
    print("\nSeçenekler:")
    print("  1) Unsloth'un 4-bit gpt-oss eğitim yolu (ayrı ortam, ayrı doğrulama)")
    print("  2) Daha büyük GPU (>=48 GB) ve bf16 LoRA")
    print("  3) 7-8B sınıfı bir taban modelle altyapıyı doğrulayıp ölçek büyütmek")
else:
    real_cfg = load_training_config(REPO_DIR / "configs" / "training.yaml")
    real_cfg.data.train_file = str(PROCESSED / "train.jsonl")
    real_cfg.data.eval_file = str(PROCESSED / "valid.jsonl")
    code = gtu_train.run_training(real_cfg, output_dir=TRAIN_OUT)
    print("Eğitim çıkış kodu:", code)
    if code != 0:
        print("Eğitim tamamlanmadı. Yukarıdaki tanılamayı inceleyin; "
              "yarıda kalan checkpoint'ler:", TRAIN_OUT)

## 8) Değerlendirme

İki ayrı şey:

* `--mode format`: veri biçimi/kalitesi (model kalitesi **değil**).
* `--mode score`: model çıktılarını sabit Türkçe test setine göre kural tabanlı
  puanlar. Tahminler ayrı üretilir (aşağıdaki hücre yalnızca format kontrolü
  yapar; gerçek skor için elinizde tahmin dosyası olmalı).

In [ ]:
from gokturk.evaluate import format_check, load_eval_set

eval_items = load_eval_set(REPO_DIR / "data" / "eval" / "turkish_eval.jsonl")
print(f"Değerlendirme seti: {len(eval_items)} öğe")
for category in sorted({item.category for item in eval_items}):
    n = sum(1 for item in eval_items if item.category == category)
    print(f"  {category:<24} {n}")

if (PROCESSED / "train.jsonl").exists():
    fmt = format_check(PROCESSED / "train.jsonl")
    print("\nBiçim kontrolü (model kalitesi ölçmez):")
    print(json.dumps({k: fmt[k] for k in ("records", "records_with_problems", "acceptable_ratio")},
                     ensure_ascii=False, indent=2))
else:
    print("\nİşlenmiş veri yok; biçim kontrolü atlandı.")

## 9) Çıktıları kaydet

Kaggle oturumu 12 saat sonra kapanır; `/kaggle/working` dışındaki dosyalar
kalıcı değildir. Bu hücre manifestoları listeler ve bir arşiv oluşturur.

In [ ]:
import shutil, zipfile

for path in sorted(pathlib.Path("/kaggle/working").rglob("*")):
    if path.is_file() and path.suffix in {".json", ".jsonl", ".md", ".txt"}:
        print(path, f"({path.stat().st_size} bayt)")

archive = pathlib.Path("/kaggle/working/gokturk_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [pathlib.Path(SMOKE_OUT), pathlib.Path(TRAIN_OUT), PROCESSED]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file() and "adapter" not in str(file):
                    zf.write(file, file.relative_to("/kaggle/working"))
print("\nArşiv:", archive, f"({archive.stat().st_size} bayt)")
print("Not: adaptör ağırlıkları arşive eklenmedi; kalıcılık için Kaggle Dataset/Model kullanın.")

## Bilinen sınırlamalar

* Bu notebook **eğitilmiş bir model teslim etmez**; altyapıyı ve dürüst
  ön denetimi çalıştırır.
* 2×T4, gpt-oss-20b'yi bu repodaki generik QLoRA yolu ile eğitemez (bkz.
  `docs/architecture.md`). Unsloth yolu bu repoda uygulanmamıştır.
* Değerlendirme seti küçük ve kural tabanlıdır; kamuya açık benchmark yerine
  geçmez.
* Öğretmen modellerin ürettiği veriler otomatik olarak doğru kabul edilmez;
  insan örneklemesi gerekir (bkz. `docs/architecture.md` — veri inceleme süreci).